In [87]:
import mosek
import cvxpy as cp
import numpy as np
print(cp.installed_solvers())


class SamSetup():
    
    #input parameters from the parameters table
    def __init__ (self,J,_lambda,Tij,Tb,T0,Tr,h,pai,ship_cost,z,S0_tilda_val, S_tilda_val):
        #number of fsl
        self.J = int(J)
        # Planning horizon
        self.T0 = T0
        # transportation time from warehouse to FSL
        self.Tr = list(Tr) 
        # penalty.
        self.h = np.reshape(np.array(list(h)),(1,J))
        # item shortage cost at FSL
        self.pai = np.reshape(np.array(list(pai)),(1,J))
        # poisson parameter. constant failure rate
        self._lambda = _lambda
        
        # items available at FSL
        self.S0_tilda = S0_tilda_val * np.ones((T0+Tb+1,1)) 

        
        #================new variables=================
        # shipment cost from the depot to FSL
        # /c_ij/ the shipment cost of item from the depot to FSL j.
        self.ship_cost = ship_cost
        # /z_ij/ the incremental cost of repairing item at FSL j instead of at the depot.
        self.z = np.array(z)
        # emmision lead time (time to make dicision)
        self.Tb = Tb
        # base-repair lead time until available at FSL
        self.Tij = list(Tij) 
        
        #size of matrix for item availability in warehouse
        self.X_size = max(self.Tij)+ max(self.Tr)+self.T0+self.Tb+1, self.J
        # items available at warehouse
        self.S_tild_size = self.X_size
        self.S_tilda = S_tilda_val * np.ones(self.S_tild_size) 
        self.cases = ''

    
    def cp_setup(self):
        
        self.y = cp.Variable((self.T0+self.Tb+1, self.J), integer=True)
        self.f = cp.Variable()
        self.S = cp.Variable(self.X_size)
        self.G = cp.Variable(self.X_size)
        self.cum_y = cp.Variable((self.y.shape), integer=True)
        
        #============new setup variables=======================
        # Decision variables to send to repair !from central depot! at :  
        self.S0 = cp.Variable(self.T0+self.Tb+1)
        #cummulative num of items at FSL for which a decision, where to send for repair has not been taken up to time t.
        self.S_tild_F = cp.Variable((self.Tb+1, self.J))
        
        self.x_b = cp.Variable((self.Tb+1, self.J), integer=True)  # the same FSL
        self.x_w = cp.Variable((self.Tb+1, self.J), integer=True)  # central depot
        self.cum_x_b = cp.Variable(self.x_b.shape, integer=True)
        self.cum_x_w = cp.Variable(self.x_w.shape, integer=True) 
        
        self.constraints = [self.f == sum(self.G) + sum(self.ship_cost*self.cum_y) + sum(self.z*self.cum_x_b),
                    self.y >= 0,
                    self.cum_y >= 0,
                    self.x_b >= 0,
                    self.x_w >= 0,
                    self.cum_x_b >= 0,
                    self.cum_x_w >= 0,
                    self.S >= 0,
                    self.S0 >=0,
                    self.G >= 0,
                    self.S_tild_F>=0]
        return self.y, self.f, self.S, self.G, self.cum_y, self.constraints, self.x_b, self.x_w, self.S_tild_F
    
    
    def sam_problem(self):
        self.cp_setup()

        def constraints_count():
            # cum_x_b, cum_x_w
            for j in range(self.J):
                for t in range(self.Tb+1):
                
                    self.constraints += [self.cum_x_b[t,j] == cp.sum(self.x_b[0:t,j])]
                    self.constraints += [self.cum_x_w[t,j] == cp.sum(self.x_w[0:t,j])]
                    self.constraints += [self.cum_x_b[t,j] + self.cum_x_w[t,j] == self.S_tild_F[t,j]] 
            # constraint 2!!! 

            # constraint 3!!
            for t in range(self.Tb+self.T0+1):
                for j in range(self.J):
                    self.constraints += [self.cum_y[t,j] == cp.sum(self.y[0:t,j])]
                self.constraints += [sum(self.cum_y[t,:]) <= self.S0[t]]
            return constraints_count 
        
        """Sould I loop over J????? """
        def constraints_S0():
            # constraint 4!!
            # constant (not affected by the failed items that arrive from FSL)
            for t in range(self.T0):
                self.constraints += [self.S0[t] == self.S0_tilda[t,0]]
            # affected by failed items that arrive from FSL 
            for t in range(self.T0, self.T0+self.Tb+1):
                self.constraints += [self.S0[t] == self.S0_tilda[self.T0 - 1, 0] + sum(self.cum_x_w[t-self.T0, :])]
            return constraints_S0
        
        """Cases functions"""
        def S_case_1(j):
            for t in range(self.Tij[j]): 
                self.constraints += [self.S[t,j] == self.S_tilda[t,j]]
            
            for t in range(self.Tij[j], self.Tij[j] + self.Tb):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_x_b[t - self.Tij[j] ,j]]
            
            for t in range(self.Tij[j] + self.Tb, self.Tr[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_x_b[self.Tb-1, j]]
            
            for t in range(self.Tr[j], self.Tr[j] + self.T0+ self.Tb):    
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tr[j]-1,j]+ self.cum_x_b[self.Tb,j] 
                                     + self.cum_y[t-self.Tr[j], j]]
            return self.constraints
        
        def S_case_2(j):
            for t in range(self.Tij[j]): 
                self.constraints += [self.S[t,j] == self.S_tilda[t,j]]

            for t in range(self.Tij[j], self.Tr[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_x_b[t - self.Tij[j] ,j]]
            
            for t in range(self.Tr[j], self.Tij[j]+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tr[j]-1,j]+ self.cum_x_b[t-self.Tij[j] ,j] 
                                     + self.cum_y[t-self.Tr[j], j]]       
                
            for t in range(self.Tij[j]+self.Tb, self.Tr[j]+self.T0+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tr[j]-1,j]+ self.cum_x_b[self.Tb,j] 
                                     + self.cum_y[t-self.Tr[j], j]] 
            return self.constraints
        
        def S_case_3(j):
            for t in range(self.Tr[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j]]
                
            for t in range(self.Tr[j], self.Tij[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_y[t-self.Tr[j],j]]
                
            for t in range(self.Tij[j], self.Tij[j]+ self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tij[j]-1,j]+ self.cum_x_b[t-self.Tij[j],j] 
                                     + self.cum_y[t-self.Tr[j],j]]
                
            for t in range(self.Tij[j]+self.Tb, self.Tr[j] + self.T0 + self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tij[j]-1,j] + self.cum_x_b[self.Tb-1, j] 
                                     + self.cum_y[t-self.Tr[j], j]]
            return self.constraints
        
        def S_case_4(j):
            for t in range(self.Tr[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j]]
                
            for t in range(self.Tr[j], self.Tr[j]+self.T0+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_y[t-self.Tr[j], j]]
                
            for t in range(self.Tr[j]+self.T0+self.Tb, self.Tij[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_y[self.T0+self.Tb, j]]

            for t in range(self.Tij[j], self.Tij[j]+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tij[j]-1,j] + self.cum_x_b[t-self.Tij[j],j] 
                                     + self.cum_y[self.T0+self.Tb,j]] 
            return self.constraints
        
        def S_case_5(j):
            for t in range(self.Tr[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j]]
                
            for t in range(self.Tr[j], self.Tij[j]):
                self.constraints += [self.S[t,j] == self.S_tilda[t,j] + self.cum_y[t-self.Tr[j],j]]
                
            for t in range(self.Tij[j], self.Tr[j]+self.T0+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tij[j]-1,j] + self.cum_y[t-self.Tr[j],j]
                                    + self.cum_x_b[t-self.Tij[j],j]]
                
            for t in range(self.Tr[j]+self.T0+self.Tb+1, self.Tij[j]+self.Tb+1):
                self.constraints += [self.S[t,j] == self.S_tilda[self.Tij[j]-1,j]+ self.cum_x_b[t-self.Tij[j],j] 
                                     + self.cum_y[self.T0+self.Tb,j]]
            return self.constraints
        
        # Calculating supply and demand constraints
        def constraints_of_S():
            for j in range(self.J):
                if ((self.Tr[j] + self.T0) >= self.Tij[j]) and ((self.Tij[j] + self.Tb) <= self.Tr[j]):
                    S_case_1(j)
                    self.cases = 'case 1'
                elif ((self.Tr[j]+self.T0) >= self.Tij[j])and((self.Tij[j] <= self.Tr[j])and(self.Tr[j]<(self.Tij[j]+self.Tb))):
                    S_case_2(j)
                    self.cases = 'case 2'
                elif ((self.Tr[j]+self.T0) >= self.Tij[j]) and ((self.Tij[j]> self.Tr[j]) and (self.Tij[j]<=(self.Tr[j]+self.Tb+self.T0))):
                    S_case_3(j)
                    self.cases = 'case 3'
                elif ((self.Tr[j]+self.T0) < self.Tij[j]) and (self.Tij[j] > (self.Tr[j]+self.T0+self.Tb)):
                    S_case_4(j)   
                    self.cases = 'case 4'
                elif ((self.Tr[j]+self.T0) < self.Tij[j]) and ((self.Tij[j] > self.Tr[j]) and (self.Tij[j] <= (self.Tr[j]+self.Tb+self.T0))):
                    S_case_5(j)
                    self.cases = 'case 5'
            return constraints_of_S        


        # Calculating cost function
        def constraints_of_G():
            for j in range(self.J):
                for t in range(min(self.Tr[j],self.Tij[j]), min(self.Tr[j],self.Tij[j])+self.T0 + self.Tb+1):
                    self.constraints += [self.G[t,j] >= self.h[0,j] * (self.S[t,j] - self._lambda*(t+1)),
                                         self.G[t,j] >= self.pai[0,j] * (self._lambda*(t+1) - self.S[t,j])]
            return constraints_of_G
        
        obj = cp.Minimize(self.f)
        constraints_count()
        constraints_S0()
        constraints_of_S()
        constraints_of_G()
        prob = cp.Problem(obj, self.constraints)
        prob.solve(solver=cp.MOSEK)
        
        return self.x_b.value, self.x_w.value, self.x_b.value.sum(), self.x_w.value.sum(), self.G.value, self.f.value, self.y.value, self.G.value.sum(), self.y.value.sum(), self.cases, self.S_tild_F.value,self.S0.value    

    def __repr__(self):
        return f"{self.J}, {self._lambda},{self.T0},{self.Tr},{self.Tb},{self.Tij},{self.h},{self.pai},{self.ship_cost},{self.z},{self.S0_tilda},{self.S_tilda},{self.S0},{self.x_b},{self.x_w},{self.f.value},{self.S_tild_F}"

    def variablesData(self):
        return {'J':self.J,'lambda':self._lambda,
            'T0':self.T0,'Tr':tuple(self.Tr),'Tb':self.Tb,'Tij':tuple(self.Tij),
            'h':tuple(self.h[0]),'pai':tuple(self.pai[0]),'ship_cost':self.ship_cost,'z':self.z,
            'S0tilda':self.S0_tilda,'Stilda':self.S_tilda,'S_tilda_F':self.S_tild_F.value,'S0':self.S0.value,
            'x_b':self.x_b.value, 'x_w':self.x_w.value, 'x_b_sum': self.x_b.value.sum(),'x_w_sum': self.x_w.value.sum(),
            
            'f_value':self.f.value,
            'G_value':self.G.value,'G_sum':self.G.value.sum(), 
            'y_value':self.y.value,'y_sum':self.y.value.sum(),'Cases':self.cases}
        
    pass

['ECOS', 'ECOS_BB', 'MOSEK', 'OSQP', 'SCIPY', 'SCS']


In [88]:
def generate_setup(case):
    """Random variables"""
    # Number of FSL;                                   type:int
    J = 2   
    
    # poisson parameter. constant failure rate;        type:int
    _lambda = 3
    
    # base-repair lead time until available at FSL;    type:tuple 
    Tij_ = tuple(np.random.choice([13,14,15],2))         
    
    # emmision lead time (time to make dicision);      type:int
    Tb = np.random.choice([2,3,4])
    
    # Planning horizon                                 type:int
    T0 = np.random.choice([3,4,5,10,11,12,15,16,17])                     
    
    # items available at FSL and warehouse;                          type:int    
    rand_S0_t = np.random.choice(np.arange(1,30,1).tolist())
    S0_tilda_val = np.random.choice(a=[0,rand_S0_t,100], p=[0.15,0.7,0.15])
    s_tild_option = np.random.choice(a=[0,rand_S0_t,100], p=[0.15,0.7,0.15])
    S_tilda_val = (np.random.choice(a=[s_tild_option, S0_tilda_val], p=[0.3,0.7]),
                  np.random.choice(a=[s_tild_option, S0_tilda_val], p=[0.3,0.7]))
    
    # penalty. holding in FSL instead of warehouse;    type:tuple
    h = tuple(np.random.choice([3,4,5,10,11,12,15,16,17],2))
    
    # item shortage cost at FSL;                       type:tuple  
    pai = int(np.random.choice(np.arange(50,100,10))), int(np.random.choice(np.arange(5,50,5)))
    
     #shipment cost of item from the depot to FSL;     type:int, options:array
    ship_cost_options = [0.3 if i==0 else i for i in np.arange(0,50,5)] 
    
    #incremental cost of repairing item at FSL instead of the depot; type:int, options:array
    z_option = [0.3 if i==0 else i for i in np.arange(0,100,10)]     
    
    """Conditional variables"""
    #Tr: transportation time from warehouse to FSL;        type:tuple 
    rand_array = np.arange(1,50)
    high_prob_high_num = [0]*6 + [0.1,0.1,0.3,0.5]
    high_prob_low_num = list(reversed(high_prob_high_num))
    Tij = max(Tij_)
    
    if case == 0:
        current_case = 'Case 1'
        tr_choice = rand_array[((Tij+Tb)<=rand_array) & ((Tij-T0)<=rand_array)] 
        if len(tr_choice)>=1:
            Tr = tuple(np.random.choice(tr_choice,2))
        else:
            Tr = None
        ship_cost = int(np.random.choice(ship_cost_options, 1, high_prob_low_num))
        z = int(np.random.choice(z_option, 1, high_prob_high_num))
        
    elif case == 1:
        current_case = 'Case 2'
        tr_choice = rand_array[(rand_array>= Tij) & ((Tij+Tb)>rand_array) & ((Tij-T0)<=rand_array)] 
        if len(tr_choice)>=1:
            Tr = tuple(np.random.choice(tr_choice,2))
        else:
            Tr = None
        ship_cost = int(np.random.choice(ship_cost_options, 1, high_prob_low_num))
        z = int(np.random.choice(z_option, 1))
        
    elif case == 2:
        current_case = 'Case 3'
        tr_choice = rand_array[(rand_array< Tij) & ((Tij-T0-Tb)<=rand_array) & ((Tij-T0)<=rand_array)] 
        if len(tr_choice)>=1:
            Tr = tuple(np.random.choice(tr_choice,2))
        else:
            Tr = None
        ship_cost = int(np.random.choice(ship_cost_options, 1, high_prob_low_num))
        z = int(np.random.choice(z_option, 1))
        
    elif case == 3:
        current_case = 'Case 4'        
        tr_choice = rand_array[(rand_array < (Tij-T0-Tb))] #& ((Tij-T0)>rand_array) 
        if len(tr_choice)>=1:
            Tr = tuple(np.random.choice(tr_choice,2))
        else:
            Tr = None
        ship_cost = int(np.random.choice(ship_cost_options, 1))
        z = int(np.random.choice(z_option, 1))
        
    elif case == 4:
        current_case = 'Case 5'
        tr_choice = rand_array[(rand_array< Tij) & ((Tij-T0-Tb)<=rand_array) & ((Tij-T0)>rand_array)] 
        if len(tr_choice)>=1:
            Tr = tuple(np.random.choice(tr_choice,2))
        else:
            Tr = None
        ship_cost = int(np.random.choice(ship_cost_options, 1, high_prob_high_num))
        z = int(np.random.choice(z_option, 1, high_prob_low_num))
        
    current_setups = {'J':J,'lambda':_lambda,'Tij':Tij_,'Tb':Tb,'T0':T0,'Tr':Tr,
            'h':h,'pai':pai,'c':ship_cost,'z':z,
            'S0_tilda':S0_tilda_val, 'S_tilda':S_tilda_val, 'Cases':current_case}
    return current_setups

In [89]:
def generate_cases(num_Setups, probabilities):
    setups = pd.DataFrame(columns=['J','lambda','Tij','Tb','T0','Tr','h','pai','c','z','S0_tilda','S_tilda','Cases'])
    output_exception = setups.copy()
    cases = np.random.choice(a=5, size=num_Setups, p=probabilities)
    for case in cases: 
        setup = generate_setup(case)
        if setup['Tr'] != None:
            setups = setups.append(setup,ignore_index=True)
        else:
            output_exception = output_exception.append(setup,ignore_index=True)
    return setups, output_exception

In [92]:
from datetime import datetime
import pandas as pd
pd.set_option('display.max_colwidth', None)

start_time = datetime.now()

inputs, exceptions = generate_cases(10, [0.2, 0.2, 0.45, 0.05, 0.1])
print(inputs.shape, exceptions.shape)
end_time = datetime.now()
print('Duration: {}'.format(end_time - start_time))

(10, 13) (0, 13)
Duration: 0:00:00.110782


In [95]:
from datetime import datetime

start_time = datetime.now()
output = pd.DataFrame(columns = ['fsl_J','failure_rate_lambda',
                    'plan_horizon_TO','transp_time_Tr', 'decision_time_Tb', 'base_repair_Tij', 
                    'penalty_h','shortage_cost_pai', 'ship_cost_c', 'increm_cost_z', 
                    'items_fsl_S0tilde',  'items_warehouse_Stilde','S_tilda_F','S0',
                    'repair_FSL_x_b', 'repair_depot_x_w', 'x_b_sum','x_w_sum','f_value','G_value','G_sum',
                    'y_value','Optimal #items (y_sum)','Cases'])

ds_len = inputs.shape[0]
for i in range(ds_len):
    if i%100 == 0:
        print(f"Processing setup # {i}")    
    setupInit = SamSetup(*tuple(inputs.iloc[i,:].values[:-1]))
    
    setupInit.sam_problem()
    var = setupInit.variablesData()
    data = {'fsl_J':var['J'],'failure_rate_lambda':var['lambda'],
            'plan_horizon_TO':var['T0'],'transp_time_Tr':var['Tr'],'decision_time_Tb':var['Tb'],'base_repair_Tij':var['Tij'],
            'penalty_h':var['h'],'shortage_cost_pai':var['pai'], 'ship_cost_c':var['ship_cost'],'increm_cost_z':var['z'],
            'items_fsl_S0tilde':var['S0tilda'],'items_warehouse_Stilde':var['Stilda'],'S_tilda_F':var['S_tilda_F'],
            'S0':var['S0'],
            'repair_FSL_x_b':var['x_b'], 'repair_depot_x_w':var['x_w'],'x_b_sum':var['x_b_sum'], 'x_w_sum':var['x_w_sum'],
            'f_value':var['f_value'], 'G_value':var['G_value'],'G_sum':var['G_sum'], 
            'y_value':var['y_value'], 'Optimal #items (y_sum)':var['y_sum'],'Cases': var['Cases']}

    output = output.append(data, ignore_index = True)
#     except: 
#         output_exception = output_exception.append(inputs.iloc[i,:], ignore_index = False)
    
end_time = datetime.now()
print('Duration: {}'.format(end_time - start_time))

Processing setup # 0
Duration: 0:00:07.210272


In [11]:
output.to_csv('SAM3_all cases_5e+4_sample_9.csv')

In [126]:
# df = output.sample(1)
# ind = df.index.tolist()[0]
# for col in df.columns:
#     try:
#         print("\n\n")
#         print(col, len(df.loc[ind,col]))
#         print(col, df.loc[ind,col].shape)
#         print(df.loc[ind,col].values)
#     except:
#         print(col, df.loc[ind,col])
